# 📈 InvestAI — Módulo de Ingesta de Datos OHLCV

**Proyecto:** Sistema Ernesto Investing AI  
**Módulo:** Ingesta de Datos — Tickers Mineros Peruanos  
**Fuente de datos:** Yahoo Finance (`yfinance`)  
**Persistencia:** MongoDB Atlas (`pymongo`)  

---

| Ticker | Empresa | Moneda |
|--------|---------|--------|
| `FSM` | Fortuna Silver Mines Inc. | USD |
| `VOLCABC1.LM` | Volcan Compañía Minera S.A.A. | PEN |
| `ABX.TO` | Barrick Gold Corporation | CAD |
| `BVN` | Compañía de Minas Buenaventura S.A.A. | USD |
| `BHP` | BHP Group Limited | USD |

**Indicadores calculados:** SMA-20, SMA-50, EMA-12, EMA-26, RSI-14  
**Colección destino:** `precios_ohlcv` (MongoDB Atlas)

> ⚠️ **Requisito previo:** Configura la variable `MONGO_URI` en los Secrets de Colab  
> (`🔑 Llave` en el panel izquierdo → "Agregar nuevo secreto" → nombre: `MONGO_URI`)

## Sección 1 — Instalación de Dependencias

In [ ]:
# Instalación de librerías necesarias
# pymongo[srv]: cliente de MongoDB con soporte para Atlas (conexión SRV)
# yfinance: descarga de datos históricos desde Yahoo Finance
!pip install "pymongo[srv]" yfinance --quiet

print("✅ Dependencias instaladas correctamente.")

## Sección 2 — Importación de Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yfinance as yf

from pymongo import MongoClient, UpdateOne
from pymongo.errors import BulkWriteError, ConnectionFailure

# Acceso seguro a los Secrets de Google Colab
from google.colab import userdata

print("✅ Librerías importadas correctamente.")
print(f"   numpy   {np.__version__}")
print(f"   pandas  {pd.__version__}")

## Sección 3 — Configuración Global

In [ ]:
# ─── Tickers del estudio (sector minero con operaciones en Perú) ──────────────
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

# Metadatos de cada empresa (nombre oficial y moneda de cotización)
EMPRESAS_META = {
    'FSM':         {'nombre': 'Fortuna Silver Mines Inc.',              'moneda': 'USD'},
    'VOLCABC1.LM': {'nombre': 'Volcan Compañía Minera S.A.A.',          'moneda': 'PEN'},
    'ABX.TO':      {'nombre': 'Barrick Gold Corporation',               'moneda': 'CAD'},
    'BVN':         {'nombre': 'Compañía de Minas Buenaventura S.A.A.',  'moneda': 'USD'},
    'BHP':         {'nombre': 'BHP Group Limited',                      'moneda': 'USD'},
}

# ─── Parámetros de descarga ───────────────────────────────────────────────────
PERIOD       = '1y'   # Último año de datos históricos
AUTO_ADJUST  = True   # Precios ajustados por dividendos y splits

# ─── Parámetros de los indicadores técnicos ───────────────────────────────────
SMA_VENTANAS = [20, 50]   # Medias móviles simples
EMA_VENTANAS = [12, 26]   # Medias móviles exponenciales
RSI_PERIODO  = 14         # Período estándar del RSI

# ─── Configuración de MongoDB ─────────────────────────────────────────────────
DB_NOMBRE    = 'investai'        # Base de datos
COL_NOMBRE   = 'precios_ohlcv'   # Colección de precios e indicadores

print("✅ Configuración cargada.")
print(f"   Tickers  : {TICKERS}")
print(f"   Período  : {PERIOD}")
print(f"   DB       : {DB_NOMBRE}.{COL_NOMBRE}")

## Sección 4 — Conexión a MongoDB Atlas

> La URI de conexión se lee desde los **Secrets de Colab** para no exponer credenciales en el código.  
> Formato esperado: `mongodb+srv://<usuario>:<contraseña>@<cluster>.mongodb.net/`

In [ ]:
# Leer MONGO_URI desde los Secrets de Colab (nunca hardcodeada en el código)
try:
    MONGO_URI = userdata.get('MONGO_URI')
    if not MONGO_URI:
        raise ValueError("El secret MONGO_URI está vacío.")
except Exception as e:
    raise RuntimeError(
        f"No se encontró el secret MONGO_URI: {e}\n"
        "Agrega tu URI en: Panel izquierdo → 🔑 → Agregar nuevo secreto → MONGO_URI"
    )

# Crear cliente de MongoDB con timeout razonable para Colab
try:
    cliente = MongoClient(
        MONGO_URI,
        serverSelectionTimeoutMS=10_000,  # 10 segundos de timeout
        connectTimeoutMS=10_000,
    )
    # Verificar conectividad enviando un ping al servidor
    cliente.admin.command('ping')
    print("✅ Conexión a MongoDB Atlas establecida correctamente.")
except ConnectionFailure as e:
    raise RuntimeError(
        f"No se pudo conectar a MongoDB Atlas: {e}\n"
        "Verifica que la URI sea correcta y que la IP de Colab esté en la lista blanca de Atlas."
    )

# Obtener referencia a la base de datos y la colección
db          = cliente[DB_NOMBRE]
coleccion   = db[COL_NOMBRE]

# Crear índice compuesto único (ticker + fecha) para evitar duplicados
# Si ya existe, MongoDB lo ignora silenciosamente
coleccion.create_index(
    [('ticker', 1), ('fecha', 1)],
    unique=True,
    name='idx_ticker_fecha'
)
print(f"   Base de datos : {DB_NOMBRE}")
print(f"   Colección     : {COL_NOMBRE}")
print(f"   Índice único  : ticker + fecha")

## Sección 5 — Funciones de Procesamiento

In [ ]:
def nan_a_none(valor):
    """
    Convierte NaN / Inf de Python/NumPy a None para que pymongo
    los serialice como null (JSON válido) en lugar de provocar error.

    MongoDB no acepta float('nan') ni float('inf') directamente;
    convertirlos a None es la práctica recomendada.

    Parámetros:
        valor : cualquier tipo numérico o None

    Retorna:
        float redondeado a 6 decimales, o None si es NaN/Inf/None
    """
    if valor is None:
        return None
    try:
        if math.isnan(valor) or math.isinf(valor):
            return None
        return round(float(valor), 6)
    except (TypeError, ValueError):
        return None

In [ ]:
def calcular_sma(serie: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la Media Móvil Simple (SMA) sobre una serie de precios.

    SMA_n(t) = promedio de los últimos n precios de cierre.
    Los primeros (n-1) valores devuelven NaN por insuficiencia de datos.

    Parámetros:
        serie   : pd.Series con precios de cierre
        ventana : número de períodos (días)

    Retorna:
        pd.Series con los valores de la SMA
    """
    return serie.rolling(window=ventana, min_periods=ventana).mean()


def calcular_ema(serie: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la Media Móvil Exponencial (EMA) sobre una serie de precios.

    La EMA pondera más los precios recientes mediante un factor de
    suavizado: α = 2 / (n + 1). Usa adjust=False para replicar la
    convención de plataformas financieras (sin corrección de sesgo).

    Parámetros:
        serie   : pd.Series con precios de cierre
        ventana : número de períodos (días)

    Retorna:
        pd.Series con los valores de la EMA
    """
    return serie.ewm(span=ventana, adjust=False, min_periods=ventana).mean()


def calcular_rsi(serie: pd.Series, periodo: int = 14) -> pd.Series:
    """
    Calcula el Índice de Fuerza Relativa (RSI) de Wilder.

    Algoritmo:
        1. Calcula cambios diarios de precio.
        2. Separa ganancias (cambios positivos) y pérdidas (cambios negativos).
        3. Calcula medias móviles exponenciales de Wilder (EWM con α=1/período).
        4. RSI = 100 - 100 / (1 + RS), donde RS = media_ganancias / media_pérdidas.

    Rango de salida: [0, 100]
        > 70 → sobrecompra (posible corrección bajista)
        < 30 → sobreventa (posible rebote alcista)

    Parámetros:
        serie   : pd.Series con precios de cierre
        periodo : número de períodos (estándar: 14)

    Retorna:
        pd.Series con los valores del RSI (primeros `periodo` valores = NaN)
    """
    delta     = serie.diff(1)
    ganancias = delta.clip(lower=0)   # Solo valores positivos (ganancias)
    perdidas  = (-delta).clip(lower=0) # Solo valores positivos de pérdidas

    # Media exponencial de Wilder: alpha = 1/periodo
    media_gan = ganancias.ewm(alpha=1.0/periodo, adjust=False, min_periods=periodo).mean()
    media_per = perdidas.ewm(alpha=1.0/periodo, adjust=False, min_periods=periodo).mean()

    # Evitar división por cero: cuando media_per = 0, RSI = 100
    rs  = media_gan / media_per.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return rsi


print("✅ Funciones de indicadores técnicos definidas:")
print("   · calcular_sma(serie, ventana)")
print("   · calcular_ema(serie, ventana)")
print("   · calcular_rsi(serie, periodo)")

In [ ]:
def descargar_y_procesar(ticker: str, period: str, auto_adjust: bool) -> pd.DataFrame | None:
    """
    Descarga datos OHLCV desde Yahoo Finance, corrige el MultiIndex,
    calcula indicadores técnicos y devuelve un DataFrame limpio.

    ⚠️ yfinance devuelve columnas con MultiIndex incluso para un solo
    ticker (ej. ('Close', 'FSM')). Se aplana con get_level_values(0)
    para obtener nombres simples ('Open', 'High', 'Low', 'Close', 'Volume').

    Parámetros:
        ticker      : símbolo bursátil (ej. 'BVN')
        period      : período de descarga (ej. '1y')
        auto_adjust : si True, aplica ajuste por dividendos y splits

    Retorna:
        pd.DataFrame con OHLCV + indicadores, o None si la descarga falla
    """
    try:
        # ── Paso 1: Descarga de datos ────────────────────────────────────────
        df = yf.download(
            ticker,
            period=period,
            auto_adjust=auto_adjust,
            progress=False,
            # multi_level_index=False  # Disponible en yfinance >= 0.2.37
        )

        if df is None or df.empty:
            print(f"   ⚠️  {ticker}: sin datos devueltos por Yahoo Finance.")
            return None

        # ── Paso 2: Corrección de MultiIndex ────────────────────────────────
        # yfinance puede devolver columnas como MultiIndex:
        #   ('Close', 'FSM'), ('Open', 'FSM'), ...
        # get_level_values(0) toma solo el primer nivel → 'Close', 'Open', ...
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Renombrar a nombres estándar en español (para consistencia interna)
        df = df.rename(columns={
            'Open'  : 'apertura',
            'High'  : 'maximo',
            'Low'   : 'minimo',
            'Close' : 'cierre',
            'Volume': 'volumen',
        })

        # Mantener solo las columnas OHLCV (descartar columnas extra si las hay)
        columnas_ohlcv = ['apertura', 'maximo', 'minimo', 'cierre', 'volumen']
        df = df[[c for c in columnas_ohlcv if c in df.columns]].copy()
        df.index = pd.to_datetime(df.index)

        # Eliminar filas donde el precio de cierre sea nulo o cero
        df = df[df['cierre'].notna() & (df['cierre'] > 0)]

        if len(df) < 30:
            print(f"   ⚠️  {ticker}: solo {len(df)} registros válidos — insuficiente para indicadores.")
            return None

        # ── Paso 3: Cálculo de indicadores técnicos ──────────────────────────
        cierre = df['cierre']

        # Medias Móviles Simples
        for v in SMA_VENTANAS:
            df[f'sma_{v}'] = calcular_sma(cierre, v)

        # Medias Móviles Exponenciales
        for v in EMA_VENTANAS:
            df[f'ema_{v}'] = calcular_ema(cierre, v)

        # RSI
        df[f'rsi_{RSI_PERIODO}'] = calcular_rsi(cierre, RSI_PERIODO)

        return df

    except Exception as e:
        print(f"   ❌ {ticker}: error durante la descarga/procesamiento → {e}")
        return None


print("✅ Función descargar_y_procesar() definida.")

In [ ]:
def construir_documentos(ticker: str, df: pd.DataFrame) -> list[dict]:
    """
    Convierte un DataFrame OHLCV + indicadores en una lista de documentos
    listos para insertar en MongoDB.

    Cada documento representa un día de cotización e incluye:
        - ticker, nombre de empresa, moneda
        - fecha (datetime UTC, para índices temporales en MongoDB)
        - OHLCV en campos separados
        - indicadores técnicos (NaN → None para JSON válido)
        - timestamp de ingesta

    Parámetros:
        ticker : símbolo bursátil
        df     : DataFrame procesado con OHLCV e indicadores

    Retorna:
        lista de dicts, uno por fila del DataFrame
    """
    meta       = EMPRESAS_META.get(ticker, {'nombre': ticker, 'moneda': 'USD'})
    timestamp  = datetime.now(timezone.utc)   # Momento de la ingesta
    documentos = []

    for fecha, fila in df.iterrows():
        doc = {
            # ── Identificación ─────────────────────────────────────────────
            'ticker'  : ticker,
            'nombre'  : meta['nombre'],
            'moneda'  : meta['moneda'],

            # ── Fecha en UTC (datetime nativo para consultas temporales) ────
            # Se almacena como datetime para permitir consultas de rango
            # eficientes: db.precios_ohlcv.find({'fecha': {'$gte': inicio}})
            'fecha'   : fecha.to_pydatetime().replace(tzinfo=timezone.utc),
            'fecha_str': fecha.strftime('%Y-%m-%d'),  # Legible para debugging

            # ── OHLCV ──────────────────────────────────────────────────────
            'apertura': nan_a_none(fila.get('apertura')),
            'maximo'  : nan_a_none(fila.get('maximo')),
            'minimo'  : nan_a_none(fila.get('minimo')),
            'cierre'  : nan_a_none(fila.get('cierre')),
            'volumen' : int(fila['volumen']) if pd.notna(fila.get('volumen')) else None,

            # ── Indicadores técnicos ───────────────────────────────────────
            # Los primeros N días tendrán None por período de calentamiento
            'sma_20'  : nan_a_none(fila.get('sma_20')),
            'sma_50'  : nan_a_none(fila.get('sma_50')),
            'ema_12'  : nan_a_none(fila.get('ema_12')),
            'ema_26'  : nan_a_none(fila.get('ema_26')),
            'rsi_14'  : nan_a_none(fila.get('rsi_14')),

            # ── Metadatos de ingesta ───────────────────────────────────────
            'ingestado_en': timestamp,
            'fuente'      : 'Yahoo Finance (yfinance)',
            'period'      : PERIOD,
            'auto_adjust' : AUTO_ADJUST,
        }
        documentos.append(doc)

    return documentos


def guardar_en_mongodb(ticker: str, documentos: list[dict]) -> dict:
    """
    Guarda los documentos en MongoDB usando operaciones upsert.

    Usa bulk_write con UpdateOne + upsert=True para que:
        - Si el documento (ticker, fecha) NO existe → lo inserta.
        - Si YA existe → lo actualiza con los nuevos valores.
    Esto hace que la ingesta sea idempotente: puedes ejecutar el
    notebook varias veces sin crear duplicados.

    Parámetros:
        ticker     : símbolo bursátil (solo para logging)
        documentos : lista de dicts generada por construir_documentos()

    Retorna:
        dict con estadísticas de la operación bulk
    """
    if not documentos:
        return {'insertados': 0, 'actualizados': 0, 'error': 'Sin documentos'}

    # Construir operaciones de upsert en lote
    operaciones = [
        UpdateOne(
            filter={'ticker': doc['ticker'], 'fecha': doc['fecha']},  # Clave única
            update={'$set': doc},                                       # Actualizar todos los campos
            upsert=True                                                # Insertar si no existe
        )
        for doc in documentos
    ]

    try:
        resultado = coleccion.bulk_write(operaciones, ordered=False)
        return {
            'insertados' : resultado.upserted_count,
            'actualizados': resultado.modified_count,
            'total'      : len(documentos),
        }
    except BulkWriteError as bwe:
        # Reporta errores parciales sin abortar (ordered=False continúa)
        n_errores = len(bwe.details.get('writeErrors', []))
        return {
            'insertados'  : bwe.details.get('nUpserted', 0),
            'actualizados': bwe.details.get('nModified', 0),
            'errores'     : n_errores,
            'aviso'       : f'{n_errores} documentos con error (posibles duplicados de índice)',
        }


print("✅ Funciones de persistencia en MongoDB definidas:")
print("   · construir_documentos(ticker, df)")
print("   · guardar_en_mongodb(ticker, documentos)")

## Sección 6 — Pipeline Principal de Ingesta

Ejecuta para cada ticker:
1. Descarga OHLCV desde Yahoo Finance
2. Corrige el MultiIndex de yfinance
3. Calcula SMA-20, SMA-50, EMA-12, EMA-26, RSI-14
4. Convierte NaN → None
5. Guarda en MongoDB Atlas (upsert idempotente)

In [ ]:
# Resumen acumulado de la ingesta
resumen_ingesta = []

print("=" * 60)
print("  PIPELINE DE INGESTA — InvestAI")
print(f"  Período : {PERIOD}  |  auto_adjust={AUTO_ADJUST}")
print("=" * 60)

for ticker in TICKERS:
    meta = EMPRESAS_META.get(ticker, {})
    print(f"\n📥 {ticker}  ({meta.get('nombre', '')}  {meta.get('moneda', '')})")

    # ── Paso 1 + 2 + 3: Descarga, corrección MultiIndex y cálculo de indicadores
    df = descargar_y_procesar(ticker, PERIOD, AUTO_ADJUST)

    if df is None:
        resumen_ingesta.append({'ticker': ticker, 'estado': '❌ Error en descarga', 'registros': 0})
        continue

    n_filas = len(df)
    print(f"   Registros descargados : {n_filas}")
    print(f"   Rango de fechas       : {df.index[0].date()} → {df.index[-1].date()}")
    print(f"   Último cierre         : {meta.get('moneda','USD')} {df['cierre'].iloc[-1]:.4f}")

    # Mostrar cuántos valores None habrá por período de calentamiento
    for col in [c for c in df.columns if c.startswith(('sma_', 'ema_', 'rsi_'))]:
        n_nulos = df[col].isna().sum()
        print(f"   {col:<10}: {n_filas - n_nulos} valores calculados  ({n_nulos} NaN → None)")

    # ── Paso 4 + 5: Construcción de documentos y guardado en MongoDB
    documentos = construir_documentos(ticker, df)
    stats = guardar_en_mongodb(ticker, documentos)

    print(f"   ✅ MongoDB → insertados: {stats.get('insertados', 0)}  "
          f"actualizados: {stats.get('actualizados', 0)}  "
          f"total procesados: {stats.get('total', len(documentos))}")

    if 'aviso' in stats:
        print(f"   ⚠️  {stats['aviso']}")

    resumen_ingesta.append({
        'ticker'     : ticker,
        'estado'     : '✅ OK',
        'registros'  : n_filas,
        'insertados' : stats.get('insertados', 0),
        'actualizados': stats.get('actualizados', 0),
    })

print("\n" + "=" * 60)
print("  RESUMEN FINAL")
print("=" * 60)
df_resumen = pd.DataFrame(resumen_ingesta)
print(df_resumen.to_string(index=False))
print("="*60)

## Sección 7 — Verificación Final

Consulta directa a MongoDB para confirmar que los datos quedaron correctamente almacenados.

In [ ]:
print("=" * 60)
print("  VERIFICACIÓN — Consulta a MongoDB Atlas")
print("=" * 60)

# ── 1. Total de documentos en la colección ────────────────────────────────────
total_docs = coleccion.count_documents({})
print(f"\n📦 Total de documentos en '{COL_NOMBRE}': {total_docs:,}")

# ── 2. Documentos por ticker ──────────────────────────────────────────────────
print("\n📊 Documentos por ticker:")
pipeline_conteo = [
    {'$group': {'_id': '$ticker', 'total': {'$sum': 1},
                'desde': {'$min': '$fecha_str'},
                'hasta': {'$max': '$fecha_str'},
                'ultimo_cierre': {'$last': '$cierre'}}},
    {'$sort': {'_id': 1}}
]
for doc in coleccion.aggregate(pipeline_conteo):
    print(f"   {doc['_id']:<14}  "
          f"{doc['total']:>4} docs  "
          f"{doc['desde']} → {doc['hasta']}  "
          f"último cierre: {doc['ultimo_cierre']}")

# ── 3. Muestra de un documento completo (el más reciente de BVN) ───────────────
print("\n📄 Ejemplo de documento (BVN, más reciente):")
doc_ejemplo = coleccion.find_one(
    {'ticker': 'BVN'},
    sort=[('fecha', -1)],
    projection={'_id': 0}   # Excluir el ObjectId para legibilidad
)

if doc_ejemplo:
    # Mostrar en formato tabla para mejor lectura
    for clave, valor in doc_ejemplo.items():
        # Formatear datetimes legiblemente
        if isinstance(valor, datetime):
            valor = valor.strftime('%Y-%m-%d %H:%M:%S UTC')
        print(f"   {clave:<20}: {valor}")
else:
    print("   ⚠️  No se encontró el documento de ejemplo (¿BVN se ingirió correctamente?)")

# ── 4. Verificar que no hay documentos con cierre nulo ─────────────────────────
n_cierres_nulos = coleccion.count_documents({'cierre': None})
print(f"\n✅ Documentos con cierre = null : {n_cierres_nulos}  (debe ser 0)")

# ── 5. Verificar índices creados ──────────────────────────────────────────────
print("\n🔑 Índices de la colección:")
for idx in coleccion.list_indexes():
    print(f"   {idx['name']:<30}  keys: {dict(idx['key'])}  "
          f"único: {idx.get('unique', False)}")

print("\n" + "=" * 60)
print("  ✅ Ingesta completada y verificada correctamente.")
print("=" * 60)

## Sección 8 — Cierre de Conexión

In [ ]:
# Cerrar la conexión a MongoDB de forma limpia al finalizar la sesión
cliente.close()
print("✅ Conexión a MongoDB Atlas cerrada.")
print()
print("📌 Para reconectar en otra sesión, vuelve a ejecutar desde la Sección 4.")